# Conversations, messages and multimodal input

Assistant version 2. Read [the module guide](README.md) for objectives, diagrams, costs and exercises.

Run all cells in order. These cells replay synthetic fixtures and block external traffic. They do not measure live model quality. Each example's code is editable below; its original command remains in the docstring. Use the CLI without `--mock` only after reviewing provider setup and cost.


In [ ]:
import os
import sys
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

ROOT = next(
    p
    for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "pyproject.toml").exists() and (p / "data").is_dir()
)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ["AWS_EC2_METADATA_DISABLED"] = "true"
os.environ["OTEL_SDK_DISABLED"] = "true"
os.environ["LITELLM_LOCAL_MODEL_COST_MAP"] = "True"
from common.network import offline_network


def run_lesson(entry_point, script):
    """Replay the script offline, on a fresh worker event loop when needed."""

    def work():
        previous = sys.argv
        sys.argv = [script, "--mock"]
        try:
            with offline_network():
                entry_point()
        finally:
            sys.argv = previous

    with ThreadPoolExecutor(max_workers=1) as pool:
        pool.submit(work).result(timeout=120)

## 01_multi_turn

Source: [01_multi_turn.py](examples/01_multi_turn.py). Predict the output, run the cell, then inspect the messages and metrics.


In [ ]:
"""Text content blocks, agent.messages, multi-turn state and role/constraints/tone/output prompts.

Run: uv run python modules/03_conversations/examples/01_multi_turn.py --mock
Expected: Lesson complete.
Cost: mock/local $0 hosted inference; live hosted calls use tokens.
AWS required: No. Resources created: none.
"""

from strands import Agent

from common.lessons import complete, lesson_settings
from common.models import get_model
from common.pretty import console, run_cli


def main() -> None:
    _, cfg = lesson_settings()
    agent = Agent(
        model=get_model(
            cfg, responses=["I noted Lisbon and a $600 budget.", "Your destination is Lisbon."]
        ),
        system_prompt="Role: Nimbus planner. Constraints: invent no availability. "
        "Tone: concise. Format: destination, budget, next question.",
        callback_handler=None,
        retry_strategy=None,
    )
    agent("Plan Lisbon with a $600 budget.", limits={"turns": 2})
    # KEY IDEA: text content blocks and earlier messages travel with the next request.
    result = agent([{"text": "Which destination did I choose?"}], limits={"turns": 2})
    console.print("Message roles:", [m["role"] for m in agent.messages])
    assert len(agent.messages) == 4
    complete(result, cfg)


run_lesson(main, "modules/03_conversations/examples/01_multi_turn.py")

## 02_conversation_windows

Source: [02_conversation_windows.py](examples/02_conversation_windows.py). Predict the output, run the cell, then inspect the messages and metrics.


In [ ]:
"""SlidingWindowConversationManager, SummarizingConversationManager, NullConversationManager and
overflow.

Run: uv run python modules/03_conversations/examples/02_conversation_windows.py --mock
Expected: Lesson complete.
Cost: mock/local $0 hosted inference; live hosted calls use tokens.
AWS required: No. Resources created: none.
"""

from strands import Agent
from strands.agent.conversation_manager import (
    NullConversationManager,
    SlidingWindowConversationManager,
    SummarizingConversationManager,
)
from strands.types.exceptions import ContextWindowOverflowException

from common.lessons import complete, lesson_settings
from common.models import get_model
from common.pretty import console, run_cli


def history() -> list[dict]:
    return [
        {"role": role, "content": [{"text": text}]}
        for _ in range(4)
        for role, text in [("user", "Lisbon; budget $600"), ("assistant", "Noted.")]
    ]


def main() -> None:
    _, cfg = lesson_settings()
    sliding = SlidingWindowConversationManager(window_size=4)
    summarizing = SummarizingConversationManager(preserve_recent_messages=2)
    # KEY IDEA: trimming discards detail; summarizing calls a model and may lose detail too.
    for manager in (sliding, summarizing, NullConversationManager()):
        model = get_model(cfg, responses=["Traveler chose Lisbon with budget $600."])
        agent = Agent(
            model=model,
            messages=history(),
            conversation_manager=manager,
            callback_handler=None,
            retry_strategy=None,
        )
        if isinstance(manager, SlidingWindowConversationManager):
            manager.apply_management(agent)
        elif isinstance(manager, SummarizingConversationManager):
            manager.reduce_context(agent)
        else:
            try:
                manager.reduce_context(agent, ContextWindowOverflowException("Context too large"))
            except ContextWindowOverflowException as exc:
                console.print("Expected overflow:", str(exc))
        console.print(type(manager).__name__, "messages remaining:", len(agent.messages))
    complete()


run_lesson(main, "modules/03_conversations/examples/02_conversation_windows.py")

## 03_receipt_image

Source: [03_receipt_image.py](examples/03_receipt_image.py). Predict the output, run the cell, then inspect the messages and metrics.


In [ ]:
"""Image content blocks and vision capability checks on a synthetic receipt.

Run: uv run python modules/03_conversations/examples/03_receipt_image.py --mock
Expected: Lesson complete.
Cost: mock/local $0 hosted inference; live hosted calls use tokens.
AWS required: No; vision model needed. Resources created: none.
"""

import os

from strands import Agent

from common.config import ROOT, load_settings
from common.data import data_path
from common.lessons import complete, lesson_settings
from common.models import get_model
from common.pretty import console, run_cli


def main() -> None:
    args, cfg = lesson_settings()
    if cfg.provider == "ollama":
        env = {**os.environ, "MODEL_ID": os.environ.get("OLLAMA_VISION_MODEL_ID", "")}
        if not env["MODEL_ID"]:
            from dotenv import dotenv_values

            env["MODEL_ID"] = dotenv_values(ROOT / ".env.example")["OLLAMA_VISION_MODEL_ID"]
        cfg = load_settings(environ=env)
    # KEY IDEA: image bytes are an image block, not text describing a file path.
    prompt = [
        {"text": "Read this synthetic receipt. Report the displayed total; do not guess."},
        {
            "image": {
                "format": "png",
                "source": {"bytes": data_path("receipts/receipt_01.png").read_bytes()},
            }
        },
    ]
    agent = Agent(
        model=get_model(cfg, responses=["Receipt RCP-001 displays USD 160.60."]),
        callback_handler=None,
        retry_strategy=None,
    )
    console.print("Live mode requires vision support; mock does not evaluate image recognition.")
    complete(agent(prompt, limits={"turns": 2}), cfg)


run_lesson(main, "modules/03_conversations/examples/03_receipt_image.py")

## 04_policy_document

Source: [04_policy_document.py](examples/04_policy_document.py). Predict the output, run the cell, then inspect the messages and metrics.


In [ ]:
"""Document blocks on supported models; local PDF text extraction with page provenance.

Run: uv run python modules/03_conversations/examples/04_policy_document.py --mock
Expected: Lesson complete.
Cost: mock/local $0 hosted inference; live hosted calls use tokens.
AWS required: Optional; local extraction. Resources created: none.
"""

import argparse

from pypdf import PdfReader
from strands import Agent

from common.data import data_path
from common.lessons import complete, lesson_settings, require_bedrock
from common.models import get_model
from common.pretty import run_cli


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--native", action="store_true", help="Use a Bedrock PDF content block")
    args, cfg = lesson_settings(parser)
    pdf = data_path("policies/nimbus_terms.pdf")
    if args.native:
        if not args.mock:
            require_bedrock(cfg)
        prompt = [
            {"text": "What is the refund threshold? Cite its PDF page."},
            {
                "document": {
                    "format": "pdf",
                    "name": "Nimbus policy",
                    "source": {"bytes": pdf.read_bytes()},
                }
            },
        ]
    else:
        # KEY IDEA: extracting text preserves page provenance even without native PDF support.
        text = "\n".join(
            f"[Page {i}] {p.extract_text()}" for i, p in enumerate(PdfReader(pdf).pages, 1)
        )
        prompt = "Use only this policy. Cite a page for supervisor approval.\n" + text
    result = Agent(
        model=get_model(cfg, responses=["Refunds over $500 need approval (page 3)."]),
        callback_handler=None,
        retry_strategy=None,
    )(prompt, limits={"turns": 2})
    complete(result, cfg)


run_lesson(main, "modules/03_conversations/examples/04_policy_document.py")

## trip_chat

Source: [trip_chat.py](use_case/trip_chat.py). Predict the output, run the cell, then inspect the messages and metrics.


In [ ]:
"""Keep a short trip conversation and show the retained history.

Run: uv run python modules/03_conversations/use_case/trip_chat.py --mock
Expected: a Nimbus result followed by Lesson complete.
Cost: mock/local $0 hosted inference; hosted mode uses model tokens.
AWS resources created: none. Concepts: compose this module's taught APIs.
"""

from strands import Agent
from strands.agent.conversation_manager import SlidingWindowConversationManager

from common.lessons import complete, lesson_settings
from common.models import get_model
from common.pretty import console, run_cli


def main() -> None:
    _, cfg = lesson_settings()
    manager = SlidingWindowConversationManager(window_size=4)
    agent = Agent(
        model=get_model(
            cfg,
            responses=[
                "Lisbon noted.",
                "Budget $600 noted.",
                "Two travelers, Lisbon, budget $600; please confirm dates.",
            ],
        ),
        conversation_manager=manager,
        callback_handler=None,
        retry_strategy=None,
        system_prompt="Nimbus planner: ask for dates and budget, never invent availability.",
    )
    # KEY IDEA: a message window is a memory budget, not durable storage of traveler preferences.
    for prompt in [
        "I want Lisbon.",
        "My budget is $600.",
        "Two travelers. Summarize and ask a question.",
    ]:
        result = agent(prompt, limits={"turns": 2})
        console.print("Retained messages:", len(agent.messages))
    complete(result, cfg)


run_lesson(main, "modules/03_conversations/use_case/trip_chat.py")

## Practice next

Complete the three exercises in the [module guide](README.md#exercises), then answer the [quiz](quiz.md). Run the same acceptance tests against your edited starters.
